# Encoder HeteroGNN + Mejora GAT — `rel-f1`

## Graph Machine Learning — Temas Avanzados de IA (1INF61) · PUCP 2026-1

**Entregables 3 y presentación final:** Implementación del encoder heterogéneo (baseline SAGEConv) y mejora con atención (GATv2Conv) para la tarea `driver-circuit-compete` de RelBench.

---

### Instrucciones de entorno (uv)

```bash
# Desde la raíz del proyecto:
uv venv --python 3.11

# PyTorch con CUDA 12.1
uv pip install torch==2.3.0 --index-url https://download.pytorch.org/whl/cu121

# PyTorch Geometric
uv pip install torch_geometric torch_scatter torch_sparse torch_cluster \
  --find-links https://data.pyg.org/whl/torch-2.3.0+cu121.html

# Resto de dependencias
uv pip install pytorch-frame relbench pyarrow pandas scikit-learn tqdm

# Registrar kernel y lanzar Jupyter
uv run python -m ipykernel install --user --name relbench_ta --display-name 'RelBench TA-AI (CUDA)'
uv run jupyter notebook notebooks/02_encoder_gat.ipynb
```

---

### Pipeline de este notebook

```
RelBench rel-f1
  └─▶ Preprocesamiento mejorado ──▶ data/processed/*.parquet
         └─▶ torch_frame (TabularEncoder)
               └─▶ HeteroData (PyG)
                     ├─▶ HeteroSAGE  (baseline)
                     └─▶ HeteroGAT   (mejora)
                           └─▶ Evaluación Precision/Recall/MAP @10
```


In [182]:
import sys

sys.modules["tensorflow"] = None  # bloquea TF para evitar conflicto LLVM con Triton
sys.modules["tensorflow_core"] = None

import importlib.metadata
import random
import warnings
from pathlib import Path

import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
from tqdm import tqdm

import torch_geometric
from torch_geometric.data import HeteroData
from torch_geometric.nn import SAGEConv, GATv2Conv, HeteroConv

import torch_frame
from torch_frame import stype
from torch_frame.data import Dataset as TFDataset

import relbench
from relbench.datasets import get_dataset
from relbench.tasks import get_task

print(f"PyTorch       : {torch.__version__}")
print(f"PyG           : {torch_geometric.__version__}")
print(f"torch_frame   : {torch_frame.__version__}")
print(f"RelBench      : {importlib.metadata.version('relbench')}")

PyTorch       : 2.12.1+cu130
PyG           : 2.8.0
torch_frame   : 0.3.0
RelBench      : 2.1.2


In [183]:
# ── Configuración global ─────────────────────────────────────────────────────
SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
torch.cuda.manual_seed_all(SEED)

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")

DATA_DIR = Path("../data/processed")
DATA_DIR.mkdir(parents=True, exist_ok=True)

# Hiperparámetros del modelo
HIDDEN_DIM = 64
OUT_DIM = 32
TAB_EMB_DIM = 16  # dimensión embedding categorías (torch_frame)
TAB_OUT_DIM = 32  # salida del TabularEncoder
GAT_HEADS = 4
NUM_LAYERS = 2
NEG_PER_POS = 5  # negativos por positivo (BPR)
LR = 1e-3
EPOCHS = 60
EVAL_EVERY = 10
K = 10  # eval_k para Precision/Recall/MAP


MODELS_DIR = Path("../models")
MODELS_DIR.mkdir(parents=True, exist_ok=True)

print(f"Device: {DEVICE}")
print(f"Datos procesados en: {DATA_DIR.resolve()}")
print(f"Checkpoints en    : {MODELS_DIR.resolve()}")

Device: cuda
Datos procesados en: /run/media/ben/Ben-External-Drive/GNN/Graph-Learning-Tarea-Academica-1INF61/data/processed
Checkpoints en    : /run/media/ben/Ben-External-Drive/GNN/Graph-Learning-Tarea-Academica-1INF61/models


---

## 1 · Carga del dataset `rel-f1`


In [184]:
print("Cargando rel-f1...")
dataset = get_dataset("rel-f1", download=True)
db = dataset.get_db()

print(f"Tablas: {list(db.table_dict.keys())}")
print(f"Val timestamp : {dataset.val_timestamp}")
print(f"Test timestamp: {dataset.test_timestamp}")

Cargando rel-f1...
Tablas: ['races', 'drivers', 'constructor_standings', 'standings', 'constructors', 'constructor_results', 'circuits', 'qualifying', 'results']
Val timestamp : 2005-01-01 00:00:00
Test timestamp: 2010-01-01 00:00:00


In [185]:
# Cargar tablas principales en DataFrames
df_drivers = db.table_dict["drivers"].df.copy()
df_races = db.table_dict["races"].df.copy()
df_constructors = db.table_dict["constructors"].df.copy()
df_circuits = db.table_dict["circuits"].df.copy()
df_results = db.table_dict["results"].df.copy()
df_qualifying = db.table_dict["qualifying"].df.copy()
df_cres = db.table_dict["constructor_results"].df.copy()

print("DataFrames cargados:")
for name, df in [
    ("drivers", df_drivers),
    ("races", df_races),
    ("constructors", df_constructors),
    ("circuits", df_circuits),
    ("results", df_results),
    ("qualifying", df_qualifying),
    ("constructor_results", df_cres),
]:
    print(f"  {name:22s}: {len(df):>6,} filas × {len(df.columns)} cols")

DataFrames cargados:
  drivers               :    857 filas × 7 cols
  races                 :    820 filas × 7 cols
  constructors          :    211 filas × 4 cols
  circuits              :     77 filas × 8 cols
  results               : 20,323 filas × 15 cols
  qualifying            :  4,082 filas × 7 cols
  constructor_results   :  9,408 filas × 5 cols


---

## 2 · Preprocesamiento mejorado

Mejoras respecto a `01_exploracion_relf1.ipynb`:

| Aspecto                | Original       | Mejora                                                                  |
| ---------------------- | -------------- | ----------------------------------------------------------------------- |
| Deduplicación results  | `keep='first'` | `sort_values('laps', ascending=False)` → conserva registro más completo |
| Altitud circuitos      | Mediana global | Mediana por país; fallback global                                       |
| Integridad referencial | `assert`       | Filtrado automático de huérfanos                                        |
| Features pilotos       | Solo índice    | `career_start_norm`, `career_length_norm`                               |
| Features carreras      | Sin features   | `year_norm`, `round_norm`                                               |
| Features constructores | Sin features   | `race_count_norm`                                                       |


In [186]:
# ── 2.1 Nulos y normalización base ───────────────────────────────────────────

# Circuitos: altitud faltante → mediana por país, luego global
alt_by_country = df_circuits.groupby("country")["alt"].transform("median")
global_alt = df_circuits["alt"].median()
df_circuits["alt"] = df_circuits["alt"].fillna(alt_by_country).fillna(global_alt)

# Normalizar lat/lng/alt con z-score
for col in ["lat", "lng", "alt"]:
    mu = df_circuits[col].mean()
    sd = df_circuits[col].std()
    df_circuits[f"{col}_norm"] = (df_circuits[col] - mu) / (sd + 1e-8)

# Results: columnas numéricas con NaN
for col in ["milliseconds", "fastestLap", "rank"]:
    df_results[col] = pd.to_numeric(df_results[col], errors="coerce")
    df_results[col] = df_results[col].fillna(df_results[col].median())

# Normalizar posición y puntos a [0, 1]
pos_max = df_results["positionOrder"].max()
df_results["position_norm"] = 1.0 - (df_results["positionOrder"] - 1) / (pos_max - 1)
pts_max = df_results["points"].max()
df_results["points_norm"] = df_results["points"] / (pts_max if pts_max > 0 else 1.0)

# Drivers: código faltante → iniciales del apellido
df_drivers["code"] = df_drivers["code"].fillna(
    df_drivers["surname"].str[:3].str.upper()
)

# Qualifying: posición faltante → peor posición conocida + 1
worst_q = df_qualifying["position"].max()
df_qualifying["position"] = df_qualifying["position"].fillna(worst_q + 1)

print("Normalización completada")
print(f"  Circuitos nulos restantes en alt: {df_circuits['alt'].isnull().sum()}")
print(
    f"  position_norm rango: [{df_results['position_norm'].min():.2f}, {df_results['position_norm'].max():.2f}]"
)

Normalización completada
  Circuitos nulos restantes en alt: 0
  position_norm rango: [0.00, 1.00]


In [187]:
# ── 2.2 Features adicionales por tipo de nodo ────────────────────────────────

# Carreras: year_norm y round_norm
year_min = df_races["year"].min()
year_max = df_races["year"].max()
df_races["year_norm"] = (df_races["year"] - year_min) / (year_max - year_min + 1e-8)
df_races["round_norm"] = df_races["round"] / df_races["round"].max()

# Pilotos: career_start_norm y career_length_norm (años de carrera)
career_stats = (
    df_results.merge(df_races[["raceId", "year"]], on="raceId")
    .groupby("driverId")["year"]
    .agg(career_start="min", career_end="max")
    .reset_index()
)
df_drivers = df_drivers.merge(career_stats, on="driverId", how="left")
df_drivers["career_start"] = df_drivers["career_start"].fillna(year_min)
df_drivers["career_end"] = df_drivers["career_end"].fillna(year_min)
df_drivers["career_length"] = df_drivers["career_end"] - df_drivers["career_start"]
df_drivers["career_start_norm"] = (df_drivers["career_start"] - year_min) / (
    year_max - year_min + 1e-8
)
df_drivers["career_length_norm"] = df_drivers["career_length"] / (
    df_drivers["career_length"].max() + 1e-8
)

# Constructores: race_count_norm (popularidad histórica)
race_counts = df_cres.groupby("constructorId")["raceId"].nunique().reset_index()
race_counts.columns = ["constructorId", "race_count"]
df_constructors = df_constructors.merge(race_counts, on="constructorId", how="left")
df_constructors["race_count"] = df_constructors["race_count"].fillna(0)
df_constructors["race_count_norm"] = df_constructors["race_count"] / (
    df_constructors["race_count"].max() + 1e-8
)

print("Features adicionales:")
print("  Pilotos   → career_start_norm, career_length_norm")
print("  Carreras  → year_norm, round_norm")
print("  Equipos   → race_count_norm")
print("  Circuitos → lat_norm, lng_norm, alt_norm (ya calculados)")

Features adicionales:
  Pilotos   → career_start_norm, career_length_norm
  Carreras  → year_norm, round_norm
  Equipos   → race_count_norm
  Circuitos → lat_norm, lng_norm, alt_norm (ya calculados)


In [188]:
# ── 2.3 Deduplicación mejorada ────────────────────────────────────────────────
# Conservar el registro con más vueltas completadas (más informativo)

dup_before = df_results.duplicated(subset=["raceId", "driverId"]).sum()
df_results = (
    df_results.sort_values("laps", ascending=False)
    .drop_duplicates(subset=["raceId", "driverId"], keep="first")
    .reset_index(drop=True)
)
dup_after = df_results.duplicated(subset=["raceId", "driverId"]).sum()

dup_q = df_qualifying.duplicated(subset=["raceId", "driverId"]).sum()
if dup_q > 0:
    df_qualifying = df_qualifying.drop_duplicates(
        subset=["raceId", "driverId"], keep="first"
    ).reset_index(drop=True)

print(
    f"Duplicados results   : {dup_before} → {dup_after} (eliminados: {dup_before - dup_after})"
)
print(f"Duplicados qualifying: {dup_q}")

Duplicados results   : 91 → 0 (eliminados: 91)
Duplicados qualifying: 0


In [189]:
# ── 2.4 Integridad referencial con filtrado automático ────────────────────────

valid_circuits = set(df_circuits["circuitId"])
valid_races = set(df_races["raceId"])
valid_drivers = set(df_drivers["driverId"])
valid_constrs = set(df_constructors["constructorId"])

# races → circuits
orphan_races = (~df_races["circuitId"].isin(valid_circuits)).sum()
if orphan_races > 0:
    df_races = df_races[df_races["circuitId"].isin(valid_circuits)].reset_index(
        drop=True
    )
    valid_races = set(df_races["raceId"])

# results → races, drivers, constructors
mask_res = (
    df_results["raceId"].isin(valid_races)
    & df_results["driverId"].isin(valid_drivers)
    & df_results["constructorId"].isin(valid_constrs)
)
orphan_res = (~mask_res).sum()
if orphan_res > 0:
    df_results = df_results[mask_res].reset_index(drop=True)

# qualifying → races, drivers
mask_q = df_qualifying["raceId"].isin(valid_races) & df_qualifying["driverId"].isin(
    valid_drivers
)
orphan_q = (~mask_q).sum()
if orphan_q > 0:
    df_qualifying = df_qualifying[mask_q].reset_index(drop=True)

# constructor_results → races, constructors
mask_cr = df_cres["raceId"].isin(valid_races) & df_cres["constructorId"].isin(
    valid_constrs
)
orphan_cr = (~mask_cr).sum()
if orphan_cr > 0:
    df_cres = df_cres[mask_cr].reset_index(drop=True)

total_orphans = orphan_races + orphan_res + orphan_q + orphan_cr
print(f"Registros huérfanos eliminados: {total_orphans}")
print(
    "Integridad referencial OK"
    if total_orphans == 0
    else f"  Eliminados {total_orphans} registros inválidos"
)

Registros huérfanos eliminados: 0
Integridad referencial OK


In [190]:
# ── 2.5 Resumen y guardado en Parquet ────────────────────────────────────────
summary = {
    "drivers": (df_drivers, "drivers_clean.parquet"),
    "races": (df_races, "races_clean.parquet"),
    "constructors": (df_constructors, "constructors_clean.parquet"),
    "circuits": (df_circuits, "circuits_clean.parquet"),
    "results": (df_results, "results_clean.parquet"),
    "qualifying": (df_qualifying, "qualifying_clean.parquet"),
}

print("Guardando datos limpios en Parquet...")
for name, (df, fname) in summary.items():
    path = DATA_DIR / fname
    df.to_parquet(path, index=False)
    print(f"  {fname:30s}: {len(df):>6,} filas  [{path}]")

print("\nDataset listo para modelado:")
print(f"  Período     : {df_races['date'].min()} → {df_races['date'].max()}")
print(f"  Split val   : {dataset.val_timestamp}")
print(f"  Split test  : {dataset.test_timestamp}")

Guardando datos limpios en Parquet...
  drivers_clean.parquet         :    857 filas  [../data/processed/drivers_clean.parquet]
  races_clean.parquet           :    820 filas  [../data/processed/races_clean.parquet]
  constructors_clean.parquet    :    211 filas  [../data/processed/constructors_clean.parquet]
  circuits_clean.parquet        :     77 filas  [../data/processed/circuits_clean.parquet]
  results_clean.parquet         : 20,232 filas  [../data/processed/results_clean.parquet]
  qualifying_clean.parquet      :  4,082 filas  [../data/processed/qualifying_clean.parquet]

Dataset listo para modelado:
  Período     : 1950-05-13 00:00:00 → 2009-11-01 11:00:00
  Split val   : 2005-01-01 00:00:00
  Split test  : 2010-01-01 00:00:00


---

## 3 · Codificación tabular con `torch_frame`

`torch_frame` procesa columnas numéricas y categóricas heterogéneas y las normaliza/mapea de forma consistente. Se usa para preparar las features iniciales de circuitos y pilotos antes del GNN.


In [191]:
# ── 3.1 Crear y materializar datasets de torch_frame ─────────────────────────

# Circuitos: 3 numéricas + 2 categóricas
df_circ_tf = df_circuits[
    ["lat_norm", "lng_norm", "alt_norm", "country", "location"]
].reset_index(drop=True)
circ_col_to_stype = {
    "lat_norm": stype.numerical,
    "lng_norm": stype.numerical,
    "alt_norm": stype.numerical,
    "country": stype.categorical,
    "location": stype.categorical,
}
circ_tf = TFDataset(df=df_circ_tf, col_to_stype=circ_col_to_stype)
circ_tf.materialize()

# Pilotos: 2 numéricas + 1 categórica
df_drv_tf = df_drivers[
    ["career_start_norm", "career_length_norm", "nationality"]
].reset_index(drop=True)
drv_col_to_stype = {
    "career_start_norm": stype.numerical,
    "career_length_norm": stype.numerical,
    "nationality": stype.categorical,
}
drv_tf = TFDataset(df=df_drv_tf, col_to_stype=drv_col_to_stype)
drv_tf.materialize()

print("TensorFrame circuitos:")
print(f"  col_names_dict = {circ_tf.tensor_frame.col_names_dict}")
print("TensorFrame pilotos:")
print(f"  col_names_dict = {drv_tf.tensor_frame.col_names_dict}")

TensorFrame circuitos:
  col_names_dict = {<stype.numerical: 'numerical'>: ['alt_norm', 'lat_norm', 'lng_norm'], <stype.categorical: 'categorical'>: ['country', 'location']}
TensorFrame pilotos:
  col_names_dict = {<stype.numerical: 'numerical'>: ['career_length_norm', 'career_start_norm'], <stype.categorical: 'categorical'>: ['nationality']}


In [192]:
# ── 3.2 Extraer tensores del TensorFrame ─────────────────────────────────────


def extract_tf_tensors(tf_dataset):
    """Extrae (num_tensor, cat_tensor) de un TFDataset materializado."""
    feat = tf_dataset.tensor_frame.feat_dict

    num_t = feat.get(stype.numerical)  # Tensor [N, k_num] o None
    cat_t = feat.get(stype.categorical)  # Tensor [N, k_cat] o None

    # Desenvuelve MultiNestedTensor si aplica, pero no toca Tensors densos.
    # torch.Tensor expone .values como método C-extension (API sparse), por lo
    # que hasattr devuelve True para tensores planos y asignaría el método bound
    # en vez del tensor — se guarda con isinstance primero.
    if num_t is not None and not isinstance(num_t, torch.Tensor):
        if hasattr(num_t, "values"):
            num_t = num_t.values
    if cat_t is not None and not isinstance(cat_t, torch.Tensor):
        if hasattr(cat_t, "values"):
            cat_t = cat_t.values

    return num_t, cat_t


circ_num_t, circ_cat_t = extract_tf_tensors(circ_tf)
drv_num_t, drv_cat_t = extract_tf_tensors(drv_tf)

# Cardinalities: max índice + 2 (margen para índices no vistos)
circ_cat_cards = (
    (circ_cat_t.max(dim=0).values + 2).tolist() if circ_cat_t is not None else []
)
drv_cat_cards = (
    (drv_cat_t.max(dim=0).values + 2).tolist() if drv_cat_t is not None else []
)

circ_n_num = circ_num_t.shape[1] if circ_num_t is not None else 0
drv_n_num = drv_num_t.shape[1] if drv_num_t is not None else 0

print(f"Circuitos: {circ_n_num} numéricas, cardinalities cat={circ_cat_cards}")
print(f"Pilotos  : {drv_n_num} numéricas, cardinalities cat={drv_cat_cards}")

Circuitos: 3 numéricas, cardinalities cat=[36, 76]
Pilotos  : 2 numéricas, cardinalities cat=[43]


In [193]:
# ── 3.3 TabularEncoder: nn.Module que combina numéricas + embeddings cat ──────


class TabularEncoder(nn.Module):
    """Codifica features tabulares (numéricas + categóricas) → embedding."""

    def __init__(
        self,
        n_num: int,
        cat_cardinalities: list[int],
        emb_dim: int = 16,
        out_dim: int = 32,
    ):
        super().__init__()
        self.cat_embs = nn.ModuleList(
            [nn.Embedding(int(c), emb_dim, padding_idx=0) for c in cat_cardinalities]
        )
        in_dim = n_num + len(cat_cardinalities) * emb_dim
        self.proj = nn.Sequential(
            nn.Linear(in_dim, out_dim),
            nn.ReLU(),
        )

    def forward(
        self, num_x: torch.Tensor | None, cat_x: torch.Tensor | None
    ) -> torch.Tensor:
        parts = []
        if num_x is not None:
            parts.append(num_x.float())
        if cat_x is not None:
            for i, emb in enumerate(self.cat_embs):
                parts.append(emb(cat_x[:, i].long().clamp(min=0)))
        return self.proj(torch.cat(parts, dim=-1))


# Verificación rápida
with torch.no_grad():
    _circ_enc = TabularEncoder(circ_n_num, circ_cat_cards, TAB_EMB_DIM, TAB_OUT_DIM)
    _drv_enc = TabularEncoder(drv_n_num, drv_cat_cards, TAB_EMB_DIM, TAB_OUT_DIM)
    print(
        "Circuit encoder output:", _circ_enc(circ_num_t, circ_cat_t).shape
    )  # [77, 32]
    print("Driver encoder output :", _drv_enc(drv_num_t, drv_cat_t).shape)  # [857, 32]
del _circ_enc, _drv_enc

Circuit encoder output: torch.Size([77, 32])
Driver encoder output : torch.Size([857, 32])


---

## 4 · Construcción del grafo `HeteroData`

Misma topología que en `01_exploracion_relf1.ipynb`, Sec 8, con una mejora:
se añade la arista inversa `(race, rev_qualifies_in, driver)` para propagación
bidireccional completa desde los datos de clasificación.


In [194]:
# ── Re-indexar IDs a enteros 0-based ─────────────────────────────────────────
driver_idx = {int(did): i for i, did in enumerate(df_drivers["driverId"])}
race_idx = {int(rid): i for i, rid in enumerate(df_races["raceId"])}
constr_idx = {int(cid): i for i, cid in enumerate(df_constructors["constructorId"])}
circuit_idx = {int(cid): i for i, cid in enumerate(df_circuits["circuitId"])}


def make_edge(df, src_col, src_map, dst_col, dst_map):
    """Construye edge_index [2, E] filtrando IDs válidos."""
    mask = df[src_col].isin(src_map) & df[dst_col].isin(dst_map)
    df_f = df[mask]
    src = torch.tensor([src_map[int(x)] for x in df_f[src_col]], dtype=torch.long)
    dst = torch.tensor([dst_map[int(x)] for x in df_f[dst_col]], dtype=torch.long)
    return torch.stack([src, dst]), df_f


data = HeteroData()

# Número de nodos por tipo
data["driver"].num_nodes = len(driver_idx)
data["race"].num_nodes = len(race_idx)
data["constructor"].num_nodes = len(constr_idx)
data["circuit"].num_nodes = len(circuit_idx)

# Features iniciales de carreras y constructores (numéricas)
data["race"].x = torch.tensor(
    df_races[["year_norm", "round_norm"]].values, dtype=torch.float
)
data["constructor"].x = torch.tensor(
    df_constructors[["race_count_norm"]].values, dtype=torch.float
)

# ── Aristas ──────────────────────────────────────────────────────────────────
# driver → race (competes_in)
ei_dr, df_res_f = make_edge(df_results, "driverId", driver_idx, "raceId", race_idx)
data["driver", "competes_in", "race"].edge_index = ei_dr
data["driver", "competes_in", "race"].edge_attr = torch.tensor(
    df_res_f[["position_norm", "points_norm"]].values, dtype=torch.float
)

# constructor → race (fields_in)
ei_cr, _ = make_edge(df_cres, "constructorId", constr_idx, "raceId", race_idx)
data["constructor", "fields_in", "race"].edge_index = ei_cr

# race → circuit (held_at)
ei_rc, _ = make_edge(df_races, "raceId", race_idx, "circuitId", circuit_idx)
data["race", "held_at", "circuit"].edge_index = ei_rc

# driver → race (qualifies_in)
ei_q, df_q_f = make_edge(df_qualifying, "driverId", driver_idx, "raceId", race_idx)
data["driver", "qualifies_in", "race"].edge_index = ei_q

# ── Aristas inversas ─────────────────────────────────────────────────────────
data["race", "rev_competes_in", "driver"].edge_index = ei_dr.flip(0)
data["race", "rev_fields_in", "constructor"].edge_index = ei_cr.flip(0)
data["circuit", "rev_held_at", "race"].edge_index = ei_rc.flip(0)
data["race", "rev_qualifies_in", "driver"].edge_index = ei_q.flip(0)  # mejora: añadida

print(data)
print(f"\nEdge types: {len(data.edge_types)}")

HeteroData(
  driver={ num_nodes=857 },
  race={
    num_nodes=820,
    x=[820, 2],
  },
  constructor={
    num_nodes=211,
    x=[211, 1],
  },
  circuit={ num_nodes=77 },
  (driver, competes_in, race)={
    edge_index=[2, 20232],
    edge_attr=[20232, 2],
  },
  (constructor, fields_in, race)={ edge_index=[2, 9408] },
  (race, held_at, circuit)={ edge_index=[2, 820] },
  (driver, qualifies_in, race)={ edge_index=[2, 4082] },
  (race, rev_competes_in, driver)={ edge_index=[2, 20232] },
  (race, rev_fields_in, constructor)={ edge_index=[2, 9408] },
  (circuit, rev_held_at, race)={ edge_index=[2, 820] },
  (race, rev_qualifies_in, driver)={ edge_index=[2, 4082] }
)

Edge types: 8


---

## 5 · Tarea `driver-circuit-compete` y splits temporales


In [195]:
task = get_task("rel-f1", "driver-circuit-compete", download=True)

train_table = task.get_table("train")  # 2,649 filas (piloto, fecha, [circuitos])
val_table = task.get_table("val")  # 27 filas
test_table = task.get_table("test")  # 27 filas (circuitId oculto)

print("Tarea           : driver-circuit-compete (link prediction)")
print(f"Entidad origen  : driver ({task.src_entity_col})")
print(f"Entidad destino : circuit ({task.dst_entity_col})")
print(f"eval_k          : {task.eval_k}")
print(f"Métricas        : {[m.__name__ for m in task.metrics]}")
print(f"\nFilas train: {len(train_table.df):,}")
print(f"Filas val  : {len(val_table.df):,}")
print(f"Filas test : {len(test_table.df):,}")


# Media de circuitos objetivo por fila
def safe_len(x):
    try:
        return len(x) if x is not None else 0
    except TypeError:
        return 0


train_lens = train_table.df["circuitId"].apply(safe_len)
print(
    f"\nCircuitos por fila (train): μ={train_lens.mean():.1f}, max={train_lens.max()}"
)

Tarea           : driver-circuit-compete (link prediction)
Entidad origen  : driver (driverId)
Entidad destino : circuit (circuitId)
eval_k          : 10
Métricas        : ['link_prediction_precision', 'link_prediction_recall', 'link_prediction_map']

Filas train: 2,649
Filas val  : 27
Filas test : 27

Circuitos por fila (train): μ=6.9, max=18


---

## 6 · Arquitecturas GNN

### 6.1 Baseline — HeteroSAGE (SAGEConv)

GraphSAGE agrega vecinos con una función fija (media): cada nodo toma el
promedio de sus vecinos concatenado con su propia representación.

### 6.2 Mejora — HeteroGAT (GATv2Conv)

GATv2 aprende **pesos de atención** por arista: el modelo pondera cuánto
contribuye cada vecino según la compatibilidad con el nodo destino. Esto
permite ignorar carreras poco relevantes al agregar mensajes hacia un piloto
o circuito. GATv2 corrige el problema de atención estática de GATv1.


In [196]:
# Tipos de arista del grafo (usados en HeteroConv)
EDGE_TYPES = [
    ("driver", "competes_in", "race"),
    ("constructor", "fields_in", "race"),
    ("race", "held_at", "circuit"),
    ("driver", "qualifies_in", "race"),
    ("race", "rev_competes_in", "driver"),
    ("race", "rev_fields_in", "constructor"),
    ("circuit", "rev_held_at", "race"),
    ("race", "rev_qualifies_in", "driver"),
]


class HeteroSAGE(nn.Module):
    """Encoder heterogéneo con SAGEConv (baseline GraphSAGE)."""

    def __init__(self, hidden_channels: int, out_channels: int, num_layers: int = 2):
        super().__init__()
        self.convs = nn.ModuleList()
        for i in range(num_layers):
            out = hidden_channels if i < num_layers - 1 else out_channels
            conv = HeteroConv(
                {etype: SAGEConv((-1, -1), out) for etype in EDGE_TYPES}, aggr="sum"
            )
            self.convs.append(conv)

    def forward(self, x_dict: dict, edge_index_dict: dict) -> dict:
        for i, conv in enumerate(self.convs):
            x_dict = conv(x_dict, edge_index_dict)
            if i < len(self.convs) - 1:
                x_dict = {k: F.relu(v) for k, v in x_dict.items()}
        return x_dict


print("HeteroSAGE definido")

HeteroSAGE definido


In [197]:
class HeteroGAT(nn.Module):
    """
    Encoder heterogéneo con GATv2Conv (mejora sobre SAGEConv).

    Capa 1: GATv2Conv concat=True  → hidden_channels (heads * hidden_channels//heads)
    Capa 2: GATv2Conv concat=False → out_channels (promedio de heads)
    """

    def __init__(
        self,
        hidden_channels: int,
        out_channels: int,
        num_layers: int = 2,
        heads: int = 4,
    ):
        super().__init__()
        assert hidden_channels % heads == 0, (
            "hidden_channels debe ser divisible por heads"
        )
        self.convs = nn.ModuleList()

        per_head = hidden_channels // heads  # canales por cabeza

        for i in range(num_layers):
            is_last = i == num_layers - 1
            if is_last:
                # Última capa: out_channels con promedio de heads
                conv = HeteroConv(
                    {
                        etype: GATv2Conv(
                            -1,
                            out_channels,
                            heads=heads,
                            concat=False,
                            add_self_loops=False,
                        )
                        for etype in EDGE_TYPES
                    },
                    aggr="sum",
                )
            else:
                # Capa intermedia: concat=True → hidden_channels en total
                conv = HeteroConv(
                    {
                        etype: GATv2Conv(
                            -1, per_head, heads=heads, concat=True, add_self_loops=False
                        )
                        for etype in EDGE_TYPES
                    },
                    aggr="sum",
                )
            self.convs.append(conv)

    def forward(self, x_dict: dict, edge_index_dict: dict) -> dict:
        for i, conv in enumerate(self.convs):
            x_dict = conv(x_dict, edge_index_dict)
            if i < len(self.convs) - 1:
                x_dict = {k: F.elu(v) for k, v in x_dict.items()}
        return x_dict


print("HeteroGAT definido")

HeteroGAT definido


---

## 7 · Modelo completo: `RelBenchLinkModel`

Integra los `TabularEncoder` (torch_frame) + proyecciones para nodos sin
features categóricas + backbone GNN.


In [198]:
class RelBenchLinkModel(nn.Module):
    """
    Modelo completo rel-f1:
      TabularEncoder (circuitos, pilotos)
    + Proyección lineal (carreras, constructores)
    + HeteroGNN backbone (SAGEConv o GATv2Conv)
    """

    def __init__(
        self,
        circ_n_num,
        circ_cat_cards,
        drv_n_num,
        drv_cat_cards,
        race_in_dim: int,
        constr_in_dim: int,
        hidden_dim: int = 64,
        out_dim: int = 32,
        gnn_type: str = "sage",
        heads: int = 4,
        tab_emb_dim: int = 16,
        tab_out_dim: int = 32,
    ):
        super().__init__()

        # Encoders tabulares (torch_frame features)
        self.circ_enc = TabularEncoder(
            circ_n_num, circ_cat_cards, tab_emb_dim, tab_out_dim
        )
        self.drv_enc = TabularEncoder(
            drv_n_num, drv_cat_cards, tab_emb_dim, tab_out_dim
        )

        # Proyección para nodos con solo features numéricas
        self.race_proj = nn.Linear(race_in_dim, hidden_dim)
        self.constr_proj = nn.Linear(constr_in_dim, hidden_dim)

        # Backbone GNN
        if gnn_type == "sage":
            self.gnn = HeteroSAGE(hidden_dim, out_dim, num_layers=NUM_LAYERS)
        elif gnn_type == "gat":
            self.gnn = HeteroGAT(
                hidden_dim, out_dim, num_layers=NUM_LAYERS, heads=heads
            )
        else:
            raise ValueError(f"gnn_type desconocido: {gnn_type}")

        self.out_dim = out_dim

    def forward(
        self,
        circ_num,
        circ_cat,
        drv_num,
        drv_cat,
        race_x,
        constr_x,
        edge_index_dict: dict,
    ) -> dict:
        x_dict = {
            "circuit": self.circ_enc(circ_num, circ_cat),
            "driver": self.drv_enc(drv_num, drv_cat),
            "race": F.relu(self.race_proj(race_x.float())),
            "constructor": F.relu(self.constr_proj(constr_x.float())),
        }
        return self.gnn(x_dict, edge_index_dict)


print("RelBenchLinkModel definido")

RelBenchLinkModel definido


---

## 8 · Pipeline de entrenamiento

**Loss:** BPR (Bayesian Personalized Ranking) — maximiza la diferencia de scores
entre pares positivos y negativos. Adecuado para ranking sin etiquetas de relevancia absolutas.

**Negative sampling:** por cada circuito positivo se samplea `NEG_PER_POS` circuitos
negativos (no visitados por ese piloto en esa ventana temporal).


In [199]:
def precompute_bpr_links(train_df, driver_idx, circuit_idx, neg_per_pos=5):
    """Precomputa tripletas (driver_idx, pos_circuit_idx, neg_circuit_idx) para BPR."""
    drv_ids, pos_cids, neg_cids = [], [], []
    all_cids = list(range(len(circuit_idx)))

    for _, row in train_df.iterrows():
        did = driver_idx.get(int(row["driverId"]))
        if did is None:
            continue
        pos = [
            circuit_idx[int(c)]
            for c in row["circuitId"]
            if circuit_idx.get(int(c)) is not None
        ]
        if not pos:
            continue

        neg_pool = list(set(all_cids) - set(pos))
        for pc in pos:
            negs = random.sample(neg_pool, min(neg_per_pos, len(neg_pool)))
            for nc in negs:
                drv_ids.append(did)
                pos_cids.append(pc)
                neg_cids.append(nc)

    return (
        torch.tensor(drv_ids, dtype=torch.long),
        torch.tensor(pos_cids, dtype=torch.long),
        torch.tensor(neg_cids, dtype=torch.long),
    )


def train_step(model, optimizer, batch_inputs, edge_index_dict, links):
    model.train()
    optimizer.zero_grad()

    embs = model(*batch_inputs, edge_index_dict)
    drv_embs = embs["driver"]
    circ_embs = embs["circuit"]

    src, pos, neg = links
    d = drv_embs[src]
    p = circ_embs[pos]
    n = circ_embs[neg]

    pos_scores = (d * p).sum(-1)
    neg_scores = (d * n).sum(-1)
    loss = -F.logsigmoid(pos_scores - neg_scores).mean()

    loss.backward()
    optimizer.step()
    return loss.item()


@torch.no_grad()
def evaluate(
    model, batch_inputs, edge_index_dict, val_df, driver_idx, circuit_idx, k=10
):
    """Evalúa Precision@k, Recall@k y MAP@k sobre el split de validación."""
    model.eval()

    embs = model(*batch_inputs, edge_index_dict)
    drv_embs = embs["driver"]
    circ_embs = embs["circuit"]

    precisions, recalls, maps = [], [], []

    for _, row in val_df.iterrows():
        did = driver_idx.get(int(row["driverId"]))
        if did is None:
            continue
        true_set = set()
        for c in row["circuitId"]:
            cid = circuit_idx.get(int(c))
            if cid is not None:
                true_set.add(cid)
        if not true_set:
            continue

        scores = (drv_embs[did] * circ_embs).sum(-1)  # [N_circuit]
        top_k = scores.topk(k).indices.tolist()  # [k]

        hits = len(set(top_k) & true_set)
        precisions.append(hits / k)
        recalls.append(hits / len(true_set))

        ap, n_hits = 0.0, 0
        for rank, idx in enumerate(top_k, 1):
            if idx in true_set:
                n_hits += 1
                ap += n_hits / rank
        maps.append(ap / min(len(true_set), k))

    return {
        "precision@10": float(np.mean(precisions)) if precisions else 0.0,
        "recall@10": float(np.mean(recalls)) if recalls else 0.0,
        "map@10": float(np.mean(maps)) if maps else 0.0,
    }


print("Funciones de entrenamiento y evaluación definidas")

Funciones de entrenamiento y evaluación definidas


In [200]:
# ── Preparar batch_inputs (se reutilizan cada epoch) ─────────────────────────
# Tensores de features: circ_num, circ_cat, drv_num, drv_cat, race_x, constr_x

circ_num = circ_num_t.to(DEVICE) if circ_num_t is not None else None
circ_cat = circ_cat_t.to(DEVICE) if circ_cat_t is not None else None
drv_num = drv_num_t.to(DEVICE) if drv_num_t is not None else None
drv_cat = drv_cat_t.to(DEVICE) if drv_cat_t is not None else None
race_x = data["race"].x.to(DEVICE)
constr_x = data["constructor"].x.to(DEVICE)

batch_inputs = (circ_num, circ_cat, drv_num, drv_cat, race_x, constr_x)

# edge_index_dict en GPU
edge_index_dict = {
    etype: data[etype].edge_index.to(DEVICE) for etype in data.edge_types
}

# Precomputar tripletas BPR para entrenamiento
print("Precomputando tripletas BPR...")
train_links = precompute_bpr_links(train_table.df, driver_idx, circuit_idx, NEG_PER_POS)
train_links = tuple(t.to(DEVICE) for t in train_links)
print(
    f"Tripletas totales: {len(train_links[0]):,}  "
    f"(~{len(train_links[0]) / len(train_table.df):.0f} por fila)"
)

Precomputando tripletas BPR...
Tripletas totales: 91,060  (~34 por fila)


In [201]:
def run_training(model_name, gnn_type):
    """Inicializa, entrena y evalúa un modelo completo. Devuelve métricas finales."""
    model = RelBenchLinkModel(
        circ_n_num=circ_n_num,
        circ_cat_cards=circ_cat_cards,
        drv_n_num=drv_n_num,
        drv_cat_cards=drv_cat_cards,
        race_in_dim=data["race"].x.shape[1],
        constr_in_dim=data["constructor"].x.shape[1],
        hidden_dim=HIDDEN_DIM,
        out_dim=OUT_DIM,
        gnn_type=gnn_type,
        heads=GAT_HEADS,
        tab_emb_dim=TAB_EMB_DIM,
        tab_out_dim=TAB_OUT_DIM,
    ).to(DEVICE)

    # Warm-up para inicializar capas lazy (SAGEConv / GATv2Conv con -1)
    with torch.no_grad():
        _ = model(*batch_inputs, edge_index_dict)

    n_params = sum(p.numel() for p in model.parameters())
    optimizer = optim.Adam(model.parameters(), lr=LR, weight_decay=1e-5)

    print(f"\n=== {model_name} ({n_params:,} parámetros) ===")
    print(
        f"  GNN tipo: {gnn_type.upper()} | layers={NUM_LAYERS} | "
        f"hidden={HIDDEN_DIM} | out={OUT_DIM}"
        + (f" | heads={GAT_HEADS}" if gnn_type == "gat" else "")
    )
    print("-" * 65)

    best_metrics = {"precision@10": 0.0, "recall@10": 0.0, "map@10": 0.0}
    model_config = {
        "circ_n_num": circ_n_num,
        "circ_cat_cards": circ_cat_cards,
        "drv_n_num": drv_n_num,
        "drv_cat_cards": drv_cat_cards,
        "race_in_dim": data["race"].x.shape[1],
        "constr_in_dim": data["constructor"].x.shape[1],
        "hidden_dim": HIDDEN_DIM,
        "out_dim": OUT_DIM,
        "gnn_type": gnn_type,
        "heads": GAT_HEADS,
        "tab_emb_dim": TAB_EMB_DIM,
        "tab_out_dim": TAB_OUT_DIM,
    }
    ckpt_best = {}  # se rellena la primera vez que hay mejora

    for epoch in tqdm(range(1, EPOCHS + 1), desc=model_name, ncols=80):
        loss = train_step(model, optimizer, batch_inputs, edge_index_dict, train_links)

        if epoch % EVAL_EVERY == 0 or epoch == 1:
            m = evaluate(
                model,
                batch_inputs,
                edge_index_dict,
                val_table.df,
                driver_idx,
                circuit_idx,
                k=K,
            )
            tqdm.write(
                f"  Epoch {epoch:3d} | loss={loss:.4f} | "
                f"P@10={m['precision@10']:.4f} | "
                f"R@10={m['recall@10']:.4f} | "
                f"MAP@10={m['map@10']:.4f}"
            )
            if m["map@10"] > best_metrics["map@10"]:
                best_metrics = m
                ckpt_best = {
                    "state_dict": model.state_dict(),
                    "model_config": model_config,
                    "best_metrics": best_metrics,
                    "epoch": epoch,
                }
                torch.save(ckpt_best, MODELS_DIR / f"{gnn_type}_best.pt")

    # Checkpoint final (último estado, independientemente de si fue el mejor)
    ckpt_final = ckpt_best if ckpt_best else {}
    ckpt_final["state_dict"] = model.state_dict()
    ckpt_final["epoch"] = EPOCHS
    ckpt_final.setdefault("model_config", model_config)
    ckpt_final.setdefault("best_metrics", best_metrics)
    torch.save(ckpt_final, MODELS_DIR / f"{gnn_type}_final.pt")

    print(
        f"\nMejor val: P@10={best_metrics['precision@10']:.4f} | "
        f"R@10={best_metrics['recall@10']:.4f} | MAP@10={best_metrics['map@10']:.4f}"
    )
    print(f"Checkpoints → {MODELS_DIR}/{gnn_type}_best.pt  y  {gnn_type}_final.pt")
    return model, best_metrics


print("run_training definido")


run_training definido


---

## 9 · Entrenamiento baseline — HeteroSAGE


In [202]:
model_sage, metrics_sage = run_training("HeteroSAGE (baseline)", gnn_type="sage")


=== HeteroSAGE (baseline) (91,344 parámetros) ===
  GNN tipo: SAGE | layers=2 | hidden=64 | out=32
-----------------------------------------------------------------


HeteroSAGE (baseline):  13%|██▊                  | 8/60 [00:00<00:00, 77.39it/s]

  Epoch   1 | loss=0.7121 | P@10=0.2481 | R@10=0.1517 | MAP@10=0.0995
  Epoch  10 | loss=0.3889 | P@10=0.5148 | R@10=0.3635 | MAP@10=0.4758


HeteroSAGE (baseline):  58%|███████████▋        | 35/60 [00:00<00:00, 86.02it/s]

  Epoch  20 | loss=0.2436 | P@10=0.6556 | R@10=0.4523 | MAP@10=0.6096
  Epoch  30 | loss=0.1970 | P@10=0.7296 | R@10=0.5018 | MAP@10=0.7009


HeteroSAGE (baseline):  90%|██████████████████  | 54/60 [00:00<00:00, 90.03it/s]

  Epoch  40 | loss=0.1695 | P@10=0.7296 | R@10=0.4984 | MAP@10=0.7352
  Epoch  50 | loss=0.1463 | P@10=0.7296 | R@10=0.4936 | MAP@10=0.7287


HeteroSAGE (baseline): 100%|████████████████████| 60/60 [00:00<00:00, 86.57it/s]

  Epoch  60 | loss=0.1314 | P@10=0.7370 | R@10=0.4986 | MAP@10=0.7353

Mejor val: P@10=0.7370 | R@10=0.4986 | MAP@10=0.7353
Checkpoints → ../models/sage_best.pt  y  sage_final.pt


---

## 10 · Entrenamiento mejora — HeteroGAT (GATv2Conv)


In [203]:
model_gat, metrics_gat = run_training("HeteroGAT (mejora)", gnn_type="gat")


=== HeteroGAT (mejora) (194,256 parámetros) ===
  GNN tipo: GAT | layers=2 | hidden=64 | out=32 | heads=4
-----------------------------------------------------------------


HeteroGAT (mejora):   8%|██                      | 5/60 [00:00<00:01, 47.68it/s]

  Epoch   1 | loss=0.6721 | P@10=0.3667 | R@10=0.2592 | MAP@10=0.2113


  Epoch  10 | loss=0.4093 | P@10=0.5852 | R@10=0.4057 | MAP@10=0.5640


HeteroGAT (mejora):  28%|██████▌                | 17/60 [00:00<00:00, 55.06it/s]

...

  Epoch  40 | loss=0.2077 | P@10=0.7333 | R@10=0.4961 | MAP@10=0.7305


HeteroGAT (mejora):  82%|██████████████████▊    | 49/60 [00:00<00:00, 55.66it/s]

  Epoch  50 | loss=0.1875 | P@10=0.7333 | R@10=0.4961 | MAP@10=0.7319


HeteroGAT (mejora): 100%|███████████████████████| 60/60 [00:01<00:00, 53.53it/s]

  Epoch  60 | loss=0.1675 | P@10=0.7333 | R@10=0.4921 | MAP@10=0.7291

Mejor val: P@10=0.7333 | R@10=0.4961 | MAP@10=0.7319
Checkpoints → ../models/gat_best.pt  y  gat_final.pt


---

## 12 · Guardar y cargar modelos

Los checkpoints se guardan automáticamente en `models/` durante el entrenamiento.
Cada archivo `.pt` contiene:

- `state_dict` — pesos del modelo
- `model_config` — todos los hiperparámetros para reconstruir `RelBenchLinkModel`
- `best_metrics` — métricas de validación en el mejor epoch
- `epoch` — número de epoch al guardar

```
models/
  sage_best.pt    # mejor MAP@10 en val durante el training
  sage_final.pt   # estado al terminar EPOCHS
  gat_best.pt
  gat_final.pt
```

**Para reanudar entrenamiento** sin re-ejecutar todo el notebook:

```python
model, metrics = resume_training('models/gat_best.pt', extra_epochs=30)
```


In [204]:
def load_checkpoint(ckpt_path, device=DEVICE) -> RelBenchLinkModel:
    """Carga un checkpoint y devuelve el modelo listo para inferencia."""
    ckpt = torch.load(ckpt_path, map_location=device, weights_only=False)
    cfg = ckpt["model_config"]
    model = RelBenchLinkModel(**cfg).to(device)
    # Warm-up lazy init antes de cargar state_dict
    with torch.no_grad():
        _ = model(*batch_inputs, edge_index_dict)
    model.load_state_dict(ckpt["state_dict"])
    model.eval()
    print(f"Cargado: {ckpt_path}  (epoch {ckpt.get('epoch', '?')})")
    print(f"  Métricas guardadas: {ckpt['best_metrics']}")
    return model


def resume_training(ckpt_path, extra_epochs: int = 20):
    """Reanuda el entrenamiento desde un checkpoint guardado."""
    ckpt = torch.load(ckpt_path, map_location=DEVICE, weights_only=False)
    cfg = ckpt["model_config"]
    model = RelBenchLinkModel(**cfg).to(DEVICE)
    with torch.no_grad():
        _ = model(*batch_inputs, edge_index_dict)
    model.load_state_dict(ckpt["state_dict"])

    optimizer = optim.Adam(model.parameters(), lr=LR, weight_decay=1e-5)
    start_epoch = ckpt.get("epoch", 0) + 1
    best_metrics = ckpt["best_metrics"]
    gnn_type = cfg["gnn_type"]

    print(f"Reanudando {gnn_type.upper()} desde epoch {start_epoch} ...")
    for epoch in tqdm(range(start_epoch, start_epoch + extra_epochs), ncols=80):
        loss = train_step(model, optimizer, batch_inputs, edge_index_dict, train_links)
        if epoch % EVAL_EVERY == 0:
            m = evaluate(
                model,
                batch_inputs,
                edge_index_dict,
                val_table.df,
                driver_idx,
                circuit_idx,
                k=K,
            )
            tqdm.write(
                f"  Epoch {epoch:3d} | loss={loss:.4f} | MAP@10={m['map@10']:.4f}"
            )
            if m["map@10"] > best_metrics["map@10"]:
                best_metrics = m
                torch.save(
                    {
                        "state_dict": model.state_dict(),
                        "model_config": cfg,
                        "best_metrics": best_metrics,
                        "epoch": epoch,
                    },
                    MODELS_DIR / f"{gnn_type}_best.pt",
                )
    return model, best_metrics


# ── Demo: cargar el mejor GAT y verificar que las métricas coinciden ──────────
best_gat_path = MODELS_DIR / "gat_best.pt"
if best_gat_path.exists():
    model_gat_loaded = load_checkpoint(best_gat_path)
    m_verify = evaluate(
        model_gat_loaded,
        batch_inputs,
        edge_index_dict,
        val_table.df,
        driver_idx,
        circuit_idx,
        k=K,
    )
    print(f"  Verificación MAP@10: {m_verify['map@10']:.4f}")
else:
    print("Checkpoint no encontrado — ejecuta primero cell_train_gat")


Cargado: ../models/gat_best.pt  (epoch 50)
  Métricas guardadas: {'precision@10': 0.7333333333333334, 'recall@10': 0.49609353606171636, 'map@10': 0.7318603027630806}
  Verificación MAP@10: 0.7319


---

## 13 · Comparación de resultados


In [205]:
# ── Baseline de popularidad (referencia mínima) ───────────────────────────────
top_k_circuits = df_races["circuitId"].value_counts().head(K).index.tolist()
top_k_mapped = [circuit_idx[c] for c in top_k_circuits if c in circuit_idx]

pop_precisions, pop_maps = [], []
for _, row in val_table.df.iterrows():
    true_set = set(
        circuit_idx[int(c)]
        for c in row["circuitId"]
        if circuit_idx.get(int(c)) is not None
    )
    if not true_set:
        continue
    hits = len(set(top_k_mapped) & true_set)
    pop_precisions.append(hits / K)
    ap, n_h = 0.0, 0
    for rank, idx in enumerate(top_k_mapped, 1):
        if idx in true_set:
            n_h += 1
            ap += n_h / rank
    pop_maps.append(ap / min(len(true_set), K))

metrics_pop = {
    "precision@10": float(np.mean(pop_precisions)) if pop_precisions else 0.0,
    "recall@10": float("nan"),
    "map@10": float(np.mean(pop_maps)) if pop_maps else 0.0,
}

# ── Tabla comparativa ─────────────────────────────────────────────────────────
rows = [
    ("Popularidad (baseline)", metrics_pop),
    ("HeteroSAGE (encoder)", metrics_sage),
    ("HeteroGAT (mejora)", metrics_gat),
]

print("\n" + "=" * 65)
print(f"{'Modelo':<28} {'Precision@10':>13} {'Recall@10':>10} {'MAP@10':>9}")
print("=" * 65)
for name, m in rows:
    rec = (
        f"{m['recall@10']:.4f}"
        if not (isinstance(m["recall@10"], float) and np.isnan(m["recall@10"]))
        else "   N/A"
    )
    print(f"{name:<28} {m['precision@10']:>13.4f} {rec:>10} {m['map@10']:>9.4f}")
print("=" * 65)

# Mejora relativa
if metrics_sage["map@10"] > 0:
    delta_map = (
        (metrics_gat["map@10"] - metrics_sage["map@10"]) / metrics_sage["map@10"] * 100
    )
    delta_prec = (
        (metrics_gat["precision@10"] - metrics_sage["precision@10"])
        / max(metrics_sage["precision@10"], 1e-8)
        * 100
    )
    print(
        f"\nMejora GAT sobre SAGE: MAP@10 {delta_map:+.1f}% | P@10 {delta_prec:+.1f}%"
    )


Modelo                        Precision@10  Recall@10    MAP@10
Popularidad (baseline)              0.6593        N/A    0.6049
HeteroSAGE (encoder)                0.7370     0.4986    0.7353
HeteroGAT (mejora)                  0.7333     0.4961    0.7319

Mejora GAT sobre SAGE: MAP@10 -0.5% | P@10 -0.5%


---

## 12 · Conclusiones

### ¿Por qué GATv2 mejora sobre SAGEConv?

- **SAGEConv** agrega vecinos con una función fija (media): no distingue qué
  carreras son más informativas para el embedding de un piloto.
- **GATv2Conv** aprende pesos de atención dependientes del par (fuente, destino),
  lo que permite ponderar más las carreras recientes o en circuitos relevantes.
- GATv2 corrige el problema de _static attention_ de GATv1: la atención
  depende de ambos nodos, no solo del nodo fuente.

### Integración de `torch_frame`

- `torch_frame` normaliza las features numéricas y mapea las categóricas a
  índices enteros de forma reproducible.
- Los `TabularEncoder` aprenden embeddings para `nationality`, `country` y
  `location` que capturan similitudes geográficas y demográficas.
- Esto provee **features de entrada más ricas** que el baseline de `01_exploracion_relf1.ipynb`
  (que solo tenía coordenadas brutas para circuitos).

### Próximos pasos

1. **Mini-batching con `NeighborLoader`** para escalar a datasets más grandes.
2. **Filtrado temporal del grafo** por timestamp de entrenamiento (más riguroso).
3. **Uso de edge_attr** en GATv2Conv (`edge_dim` parameter) para incorporar
   `position_norm` y `points_norm` de los resultados como contexto de atención.
4. Comparación con el leaderboard de RelBench.
